# FairFace gender classification — M1 & M2

Notebook này clone project từ GitHub, cài requirements, tải và cache FairFace margin 1.25, chạy training DDP trên 2×T4, sau đó lưu metrics và checkpoint. Chọn **GPU T4 x2**, bật **Internet**, không gắn dataset vào Input.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json, shutil
import numpy as np, pandas as pd

SEED = 42
FAIRFACE_MARGIN = "1.25"
STORE_SIZE = 224
BS_PER_GPU = 128
CLEANUP_CACHE = False
MODELS_TO_RUN = ['m1', 'm2']

ROOT = Path("/kaggle/working")
REPO_URL = "https://github.com/NoBugNoPain500/Deep_Learning_V2.git"
GITHUB_BRANCH = "main"
REPO_DIR = ROOT / "Deep_Learning_V2"
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
PROJECT = REPO_DIR / "fairface_project"
assert (PROJECT / "requirements.txt").is_file(), f"Missing project: {PROJECT}"
CACHE = ROOT / f"fairface_{FAIRFACE_MARGIN}_cache"
RESULT = ROOT / "results"
CACHE.mkdir(parents=True, exist_ok=True)
RESULT.mkdir(parents=True, exist_ok=True)
print("Repository:", REPO_URL, "branch:", GITHUB_BRANCH)
print("Project:", PROJECT)
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True, check=True).stdout)

## Cài dependencies từ project

PyTorch và torchvision có CUDA được cung cấp bởi Kaggle GPU image; requirements không cài đè hai package đó.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt")], check=True)
print("Dependencies installed from", PROJECT / "requirements.txt")

## Tải dữ liệu và chuẩn bị cache

Module trong repository tải FairFace từ Hugging Face về `/kaggle/working`, resize ảnh và tạo train/dev split cố định. Validation chính thức chỉ dùng làm test.

In [ ]:
prepare_cmd = [sys.executable, "-m", "fairface.data", "--root", str(ROOT), "--margin", FAIRFACE_MARGIN, "--size", str(STORE_SIZE)]
subprocess.run(prepare_cmd, cwd=PROJECT, check=True)
print("Cache size (GB):", round(sum(p.stat().st_size for p in CACHE.glob("*.npy")) / 1024**3, 2))

## Sanity check và huấn luyện

Sanity check chạy trong `torchrun`. Sau đó từng model được chạy trong tiến trình riêng. Khi cần resume sau khi session dừng, chạy lại notebook trên cùng workspace: project CLI tiếp tục từ checkpoint `*_last.pt`.

In [ ]:
env = os.environ.copy()
env.update({"NCCL_IB_DISABLE": "1", "OMP_NUM_THREADS": "2", "PYTHONUNBUFFERED": "1"})
def run(model_args):
    cmd = ["torchrun", "--standalone", "--nproc_per_node=2", "-m", "fairface.train"] + model_args
    print("RUN:", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=PROJECT, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc:
        raise RuntimeError(f"torchrun exited with status {rc}")

run(["--sanity", "--model", MODELS_TO_RUN[0], "--cache", str(CACHE), "--out", str(RESULT), "--bs", str(BS_PER_GPU)])
for model_id in MODELS_TO_RUN:
    if (RESULT / f"{model_id}_metrics.json").exists() and (RESULT / f"{model_id}_best.pt").exists():
        print(model_id, "already complete; skipping")
        continue
    run(["--model", model_id, "--cache", str(CACHE), "--out", str(RESULT), "--bs", str(BS_PER_GPU), "--seed", str(SEED)])

## Kết quả và artifacts

Kaggle sẽ hiển thị file trong `/kaggle/working/results` ở phần Output. Tải checkpoint và metrics về máy trước khi session kết thúc nếu cần giữ lại.

In [ ]:
rows = []
for model_id in MODELS_TO_RUN:
    metrics_path = RESULT / f"{model_id}_metrics.json"
    if metrics_path.exists():
        m = json.loads(metrics_path.read_text())
        rows.append({k: v for k, v in m.items() if k not in ("confusion_matrix", "precision_recall_by_class", "race_accuracy")})
        print("\n", model_id, "race accuracy:", m.get("race_accuracy"))
summary = pd.DataFrame(rows)
display(summary)
summary.to_csv(RESULT / "summary.csv", index=False)
print("Result files:", [p.name for p in sorted(RESULT.iterdir())])
if CLEANUP_CACHE:
    shutil.rmtree(CACHE, ignore_errors=True)
    shutil.rmtree(ROOT / "hf_cache", ignore_errors=True)
    print("Cache removed; checkpoints and results preserved")